# Cumulative Distribution Examples

This notebook demonstrates the usage of the ECDF (Empirical Cumulative Distribution Function) features from `khisto`.

## Key Features

- **`ecdf()`**: Returns a callable ECDF object with linear interpolation
- **`ecdf_values()`**: Returns discrete CDF values at bin edges
- **`ecdf_values_table()`**: Returns a DataFrame with CDF data
- **`ECDF` class**: Callable object that evaluates CDF at any point
- **`ECDFCollection`**: Collection of ECDFs for all granularity levels
- **Plotly `khisto_plotly.ecdf()`**: Visualization of CDFs with interactive features

## Setup

Import the required libraries:

In [17]:
import os

os.environ["KHISTO_BIN_DIR"] = (
    "/home/elouen/python/khisto-python/src/khiops/build/linux-gcc-debug/bin/khisto"
)

import numpy as np
import pandas as pd
from khisto.array import ecdf, ecdf_values, ecdf_values_table
import khisto.plotly as khisto_plotly

# Set random seed for reproducibility
np.random.seed(42)

## The ECDF Class

### Creating and Using an ECDF

The `ecdf()` function returns a callable `ECDF` object that can evaluate the CDF at any point using linear interpolation:

In [18]:
# Generate random data from a normal distribution
data = np.random.normal(0, 1, 1000)

# Create an ECDF object
ecdf_result = ecdf(data)
print(ecdf_result)

# Evaluate at a single point
print(f"\nCDF at x=0: {ecdf_result(0.0):.4f}")
print(f"CDF at x=-1: {ecdf_result(-1.0):.4f}")
print(f"CDF at x=1: {ecdf_result(1.0):.4f}")

ECDF(granularity=0, is_best=True, n_points=8, range=[-3.25, 3.875])

CDF at x=0: 0.5000
CDF at x=-1: 0.1542
CDF at x=1: 0.8400


In [19]:
ecdf_result(0.0)

np.float64(0.5)

### Evaluating at Multiple Points

The ECDF can be called with arrays for vectorized evaluation:

In [20]:
# Evaluate at many points (linear interpolation between bin edges)
x_points = np.linspace(-3, 3, 100)
cdf_values_at_points = ecdf_result(x_points)

# Plot the ECDF
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_points, y=cdf_values_at_points, mode="lines", name="ECDF"))
fig.update_layout(
    title="ECDF with Linear Interpolation",
    xaxis_title="Value",
    yaxis_title="Cumulative Probability",
    template="plotly_white",
)
fig.show()

### ECDF Properties

Access the underlying positions and CDF values:

In [21]:
print(f"Granularity: {ecdf_result.granularity}")
print(f"Is best: {ecdf_result.is_best}")
print(f"Number of bin edges: {len(ecdf_result.positions)}")
print(f"Range: [{ecdf_result.positions[0]:.4f}, {ecdf_result.positions[-1]:.4f}]")
print(f"\nPositions (first 5): {ecdf_result.positions[:5]}")
print(f"CDF values (first 5): {ecdf_result.cdf_values[:5]}")

Granularity: 0
Is best: True
Number of bin edges: 8
Range: [-3.2500, 3.8750]

Positions (first 5): [-3.25   -2.125  -1.5625 -0.875   0.875 ]
CDF values (first 5): [0.    0.009 0.047 0.178 0.822]


## ECDFCollection: Multiple Granularities

### Getting All Granularity Levels

Use `granularity=None` to get an `ECDFCollection` with all granularity levels:

In [22]:
# Get ECDFs for all granularity levels
cdf_collection = ecdf(data, granularity=None)
print(cdf_collection)
print(f"\nAvailable granularities: {cdf_collection.granularities}")
print(f"Best granularity: {cdf_collection.best.granularity}")

ECDFCollection(n_granularities=7, granularities=[0, 1, 2, 3, 4, 5, 6], best_granularity=6)

Available granularities: [0, 1, 2, 3, 4, 5, 6]
Best granularity: 6


### Accessing Individual Granularities

Access ECDFs by granularity level using indexing or the `.best` property:

In [23]:
# Access by index
ecdf_g0 = cdf_collection[0]
ecdf_g2 = cdf_collection[2]

# Access best
ecdf_best = cdf_collection.best

print(f"Granularity 0: {ecdf_g0}")
print(f"Granularity 2: {ecdf_g2}")
print(f"Best: {ecdf_best}")

# Compare evaluations at x=0
print("\nCDF at x=0 for different granularities:")
print(f"  Granularity 0: {ecdf_g0(0.0):.4f}")
print(f"  Granularity 2: {ecdf_g2(0.0):.4f}")
print(f"  Best: {ecdf_best(0.0):.4f}")

Granularity 0: ECDF(granularity=0, is_best=False, n_points=2, range=[-3.25, 3.875])
Granularity 2: ECDF(granularity=2, is_best=False, n_points=7, range=[-3.25, 3.875])
Best: ECDF(granularity=6, is_best=True, n_points=8, range=[-3.25, 3.875])

CDF at x=0 for different granularities:
  Granularity 0: 0.4561
  Granularity 2: 0.4960
  Best: 0.5000


### Comparing Granularity Levels

Visualize how different granularity levels affect the ECDF:

In [24]:
x_points = np.linspace(-3, 3, 200)

fig = go.Figure()
for ecdf_obj in cdf_collection:
    label = f"Granularity {ecdf_obj.granularity}"
    if ecdf_obj.is_best:
        label += " (best)"
    fig.add_trace(
        go.Scatter(
            x=x_points,
            y=ecdf_obj(x_points),
            mode="lines",
            name=label,
            line=dict(width=3 if ecdf_obj.is_best else 1),
        )
    )

fig.update_layout(
    title="ECDFs at Different Granularity Levels",
    xaxis_title="Value",
    yaxis_title="Cumulative Probability",
    template="plotly_white",
)
fig.show()

## ecdf_values: Discrete CDF Data

### Getting Raw CDF Values

Use `ecdf_values()` to get the discrete CDF values as arrays (positions and cumulative probabilities):

In [25]:
# Get discrete CDF values for best granularity
cdf_vals, positions = ecdf_values(data)
print(f"Number of points: {len(positions)}")
print(f"Positions: {positions[:5]}...")
print(f"CDF values: {cdf_vals[:5]}...")

Number of points: 8
Positions: [-3.25   -2.125  -1.5625 -0.875   0.875 ]...
CDF values: [0.    0.009 0.047 0.178 0.822]...


### All Granularities as List

Use `granularity=None` to get values for all granularity levels:

In [26]:
# Get all granularities
all_values = ecdf_values(data, granularity=None)
print(f"Number of granularity levels: {len(all_values)}")
for i, (cdf_vals, positions) in enumerate(all_values):
    print(f"  Granularity {i}: {len(positions)} points")

Number of granularity levels: 7
  Granularity 0: 2 points
  Granularity 1: 4 points
  Granularity 2: 7 points
  Granularity 3: 8 points
  Granularity 4: 9 points
  Granularity 5: 8 points
  Granularity 6: 8 points


## ecdf_values_table: DataFrame Output

### Getting CDF as a DataFrame

Use `ecdf_values_table()` to get a DataFrame with all CDF data:

In [27]:
# Get CDF as DataFrame
cdf_table = ecdf_values_table(data)
print(cdf_table.to_pandas())

    position  granularity  is_best  cumulative_probability  \
0    -3.2500            0    False                   0.000   
1     3.8750            0    False                   1.000   
2    -3.2500            1    False                   0.000   
3    -2.0000            1    False                   0.017   
4     2.0000            1    False                   0.976   
5     3.8750            1    False                   1.000   
6    -3.2500            2    False                   0.000   
7    -2.0000            2    False                   0.017   
8    -1.0000            2    False                   0.147   
9     1.0000            2    False                   0.845   
10    2.0000            2    False                   0.976   
11    3.0000            2    False                   0.998   
12    3.8750            2    False                   1.000   
13   -3.2500            3    False                   0.000   
14   -2.0000            3    False                   0.017   
15   -1.

### All Granularities in DataFrame

In [28]:
# Get all granularities
cdf_table_all = ecdf_values_table(data, granularity=None)
df = cdf_table_all.to_pandas()
print(f"Total rows: {len(df)}")
print(f"Unique granularities: {df['granularity'].unique()}")
print(df.head(10))

Total rows: 46
Unique granularities: [0 1 2 3 4 5 6]
   position  granularity  is_best  cumulative_probability  \
0    -3.250            0    False                   0.000   
1     3.875            0    False                   1.000   
2    -3.250            1    False                   0.000   
3    -2.000            1    False                   0.017   
4     2.000            1    False                   0.976   
5     3.875            1    False                   1.000   
6    -3.250            2    False                   0.000   
7    -2.000            2    False                   0.017   
8    -1.000            2    False                   0.147   
9     1.000            2    False                   0.845   

   cumulative_frequency  
0                     0  
1                  1000  
2                     0  
3                    17  
4                   976  
5                  1000  
6                     0  
7                    17  
8                   147  
9              

## Plotly Visualizations

### Basic Cumulative Distribution Plot

Create a cumulative distribution plot using Plotly:

In [29]:
# Basic cumulative distribution plot
fig = khisto_plotly.ecdf(
    x=data, title="Cumulative Distribution", template="plotly_white"
)
fig.show()

### Comparing Multiple Groups

In [30]:
# Create dataset with multiple categories
df = pd.DataFrame(
    {
        "value": np.concatenate(
            [
                np.random.normal(-1, 0.8, 500),  # Group A
                np.random.normal(0, 1, 500),  # Group B
                np.random.normal(1, 1.2, 500),  # Group C
            ]
        ),
        "group": ["A"] * 500 + ["B"] * 500 + ["C"] * 500,
    }
)

fig = khisto_plotly.ecdf(
    df,
    x="value",
    color="group",
    title="Comparing Multiple Distributions",
    template="plotly_white",
)
fig.show()

### Interactive Granularity Slider

Use `granularity=None` for interactive exploration:

In [31]:
fig = khisto_plotly.ecdf(
    x=data,
    granularity=None,  # Enable interactive slider
    title="Interactive Granularity Explorer",
    template="plotly_white",
)
fig.show()

## Practical Example: Comparing ECDF vs Scipy

Compare khisto's ECDF with scipy's empirical CDF:

In [32]:
from scipy.stats import ecdf as scipy_ecdf

# Create khisto ECDF
khisto_cdf = ecdf(data)

# Create scipy ECDF
scipy_cdf = scipy_ecdf(data).cdf

# Compare at same points
x_eval = np.linspace(-3, 3, 200)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_eval, y=khisto_cdf(x_eval), mode="lines", name="Khisto ECDF (optimal bins)"
    )
)
fig.add_trace(
    go.Scatter(
        x=x_eval,
        y=scipy_cdf.evaluate(x_eval),
        mode="lines",
        name="Scipy ECDF (step function)",
    )
)
fig.update_layout(
    title="Khisto vs Scipy ECDF",
    xaxis_title="Value",
    yaxis_title="Cumulative Probability",
    template="plotly_white",
)
fig.show()

## Summary

The khisto ECDF API provides:

| Function | Returns | Description |
|----------|---------|-------------|
| `ecdf(x)` | `ECDF` | Callable object with linear interpolation |
| `ecdf(x, granularity=None)` | `ECDFCollection` | All granularity levels |
| `ecdf_values(x)` | `(cdf_values, positions)` | Discrete CDF points |
| `ecdf_values_table(x)` | `DataFrame` | CDF data as table |
| `khisto_plotly.ecdf(x)` | `plotly.Figure` | Interactive visualization |

Key features:
- **Linear interpolation**: Evaluate CDF at any point, not just bin edges
- **Multiple granularities**: Access different resolution levels
- **Callable interface**: Use ECDF like scipy's `ecdf().cdf`
- **Backend-agnostic**: Works with NumPy, lists, pandas, polars